# PDC OS-Fed: run FedAvg (Step 7, run c) on Colab
Run the cells **in order**. Before you start: Runtime > Change runtime type > **T4 GPU**.
Add a Colab Secret named `GITHUB_TOKEN` only if your repository is private (key icon in the left bar).
Results are saved on your Google Drive, so a disconnect does not lose them: use the *Resume* cell.

## 1. Check the GPU

In [ ]:
import torch, subprocess
print("torch", torch.__version__, "| GPU available:", torch.cuda.is_available())
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)

## 2. Connect Google Drive (results and checkpoints are stored here)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Get the code from GitHub  (edit GITHUB_USER)

In [ ]:
import os
GITHUB_USER = "YOUR_GITHUB_USERNAME"
REPO = "pdc-osfed"
try:
    from google.colab import userdata
    TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    TOKEN = ""
os.chdir("/content")
if not os.path.exists(f"/content/{REPO}"):
    auth = f"{TOKEN}@" if TOKEN else ""
    !git clone https://{auth}github.com/{GITHUB_USER}/{REPO}.git
os.chdir(f"/content/{REPO}")
!git log --oneline | head -5
!ls

## 4. Store logs, checkpoints and the CIFAR-10 cache on Drive

In [ ]:
import os, shutil
PROJECT = f"/content/{REPO}"
OUT = "/content/drive/MyDrive/pdc-osfed-out"
for sub, drive_name in [("logs", "logs"), ("checkpoints", "checkpoints"), ("data/raw", "data_raw")]:
    target = f"{OUT}/{drive_name}"
    os.makedirs(target, exist_ok=True)
    link = f"{PROJECT}/{sub}"
    os.makedirs(os.path.dirname(link), exist_ok=True)
    if os.path.islink(link):
        os.remove(link)
    elif os.path.isdir(link):
        shutil.rmtree(link)
    os.symlink(target, link)
    print(link, "->", target)

## 5. Unit checks and environment file

In [ ]:
%cd /content/pdc-osfed
!pip -q install pyyaml
!python -m scripts.test_fedavg
!python -m scripts.save_env --name colab

## 6. Partition files: they must be the SAME as on Ubuntu
Compare these fingerprints with the ones on Ubuntu (command in the chat).

In [ ]:
import json, glob, subprocess
files = sorted(glob.glob("data/partitions/*.json"))
if not files:
    print("No partition files in the repo: generating them (downloads CIFAR-10 first time).")
    subprocess.run(["python", "-m", "src.partition"], check=True)
    files = sorted(glob.glob("data/partitions/*.json"))
for p in files:
    print(p, json.load(open(p))["meta"]["fingerprint"][:16])

## 7. Timing test: 10 clients, 3 rounds (read the seconds per round)

In [ ]:
!python -m src.run_single --config configs/A_fedavg_cnn_noniid_r3_s1.yaml

## 8. Choose the settings for the full run
Total time = seconds per round (from step 7) x rounds. The paper uses 100 rounds and 5 local epochs.
If that fits in about 1 to 2 hours, keep 100 and 5. Otherwise reduce, and write the reason in `notes/decisions.md`.
Use the **same** values for OS-Fed later.

In [ ]:
ROUNDS = 100
LOCAL_EPOCHS = 5
CFG = "configs/A_fedavg_cnn_noniid_wavg_s1.yaml"
!sed -i "s/^rounds: .*/rounds: {ROUNDS}/; s/^local_epochs: .*/local_epochs: {LOCAL_EPOCHS}/" {CFG}
!grep -E "^(rounds|local_epochs|ckpt_every|seed|aggregation|batch_size|lr):" {CFG}

## 9. Full run (c). Keep this browser tab open. If it stops, use cell 10.

In [ ]:
!python -m src.run_single --config {CFG}

## 10. Resume after a disconnect (paste the run_id printed at the start of cell 9, then run cells 1 to 4 again first)

In [ ]:
RUN_ID = "PASTE_RUN_ID_HERE"
!python -m src.run_single --resume {RUN_ID}

## 11. Look at the result

In [ ]:
import glob, pandas as pd
import matplotlib.pyplot as plt
runs = sorted(glob.glob("logs/*_A_fedavg_cnn_noniid_wavg_s1__net0"))
RUN = runs[-1]
df = pd.read_csv(f"{RUN}/rounds.csv")
print(RUN)
print(df[["round", "test_acc", "test_loss", "upload_bytes_total", "download_bytes_total", "round_wall_s"]].tail())
print("total hours:", round(df["round_wall_s"].sum() / 3600, 2))
df.plot(x="round", y="test_acc", title="FedAvg test accuracy"); plt.show()

## 12. Download the results (then copy them into the Ubuntu project)

In [ ]:
import shutil, os
from google.colab import files
run_id = os.path.basename(RUN)
shutil.make_archive(f"/content/{run_id}_logs", "zip", "logs", run_id)
files.download(f"/content/{run_id}_logs.zip")
files.download(f"checkpoints/{run_id}/latest.pt")
files.download("notes/env_colab.txt")
files.download("notes/run_log.md")